# DM887 GRPO for Control: Midway ObjectRL Baseline PoC

This candidate notebook is an experiment controller for the DM887 midway/interim report. It does **not** implement PPO, SAC, TD3, or the final GRPO-control algorithm from scratch. Instead, it uses ObjectRL as the baseline implementation source and prepares the reproducible pipeline needed for report-facing PPO/SAC/TD3 baseline runs.

Official assignment scope from `DM887_Project.pdf`:

- Final project goal: design, analyze, and evaluate a GRPO variant for control tasks.
- Final comparison: GRPO versus PPO, SAC, and TD3.
- Required environments: continuous Car Racing from Gymnasium/Farama Box2D, `cartpole-swingup-v0`, and `acrobot-swingup-v0` from DeepMind Control Suite.
- Required evaluation: online training, regular evaluation intervals, learning curves with undiscounted evaluation episode return on the y-axis and training steps before evaluation on the x-axis, using five seeds.

Midway/interim scope:

- Related work.
- Formal MDP notation and setup.
- PPO/SAC/TD3 baseline protocol and complete/interim baseline results.

**Important TODO:** ObjectRL currently exposes a limited DMC environment mapping. The DMC swingup IDs and Car Racing continuous ID must be verified before launching full runs. This notebook surfaces those checks explicitly.


## Project-context summary

1. **Full final goal:** Start from vanilla GRPO, design a GRPO variant suitable for control tasks, compare it against PPO, SAC, and TD3, and provide convergence-oriented analysis in a NeurIPS-style final report.
2. **Midway deliverables:** Establish related work, MDP notation, and an ObjectRL-based PPO/SAC/TD3 baseline pipeline with report-ready logs, status files, CSV summaries, and learning curves where available.
3. **Repository structure:** Project knowledge lives in `docs/`, concrete work plans in `plans/`, notebooks in `notebooks/`, generated logs/results in `results/`, figures in `figures/`, and external repositories in `external/`.
4. **Intended PoC workflow:** Inspect ObjectRL, discover model and environment names, define the baseline matrix, generate commands, dry-run first, run selected jobs through subprocess, save logs/status, parse evaluation returns, aggregate by algorithm/environment/seed/step, and export midway figures.
5. **Assumptions to verify before experiments:** ObjectRL is installed in the active Python environment, the required Gymnasium Box2D/DMC environments can be instantiated, ObjectRL CLI keys match the generated commands, and the selected training budgets fit available compute.


## Imports and robust repository paths

Run this notebook from the repository root when possible. If VS Code/Jupyter starts in `notebooks/`, the helper below walks upward until it finds the repository markers.


In [ ]:
from __future__ import annotations

import ast
import csv
import itertools
import json
import os
import re
import shlex
import subprocess
import sys
import time
from dataclasses import asdict, is_dataclass
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_repo_root(start: Path | None = None) -> Path:
    """Find the repository root from a notebook or repository working directory."""
    start = (start or Path.cwd()).resolve()
    markers = ("README.md", "DM887_Project.pdf", "external")
    for candidate in (start, *start.parents):
        if all((candidate / marker).exists() for marker in markers):
            return candidate
    raise RuntimeError(
        "Could not find the repository root. Start Jupyter from the DM887_Project root."
    )


REPO_ROOT = find_repo_root()
OBJECTRL_DIR = REPO_ROOT / "external" / "objectrl"
GYMNASIUM_DIR = REPO_ROOT / "external" / "Gymnasium"
RESULTS_DIR = REPO_ROOT / "results"
RAW_RESULTS_DIR = RESULTS_DIR / "raw"
PROCESSED_RESULTS_DIR = RESULTS_DIR / "processed"
LOGS_DIR = RESULTS_DIR / "logs"
FIGURES_DIR = REPO_ROOT / "figures" / "midway"
NOTEBOOK_RESULTS_DIR = RAW_RESULTS_DIR / "midway_poc_copilotcli"

for directory in [RAW_RESULTS_DIR, PROCESSED_RESULTS_DIR, LOGS_DIR, FIGURES_DIR, NOTEBOOK_RESULTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"REPO_ROOT: {REPO_ROOT}")
print(f"OBJECTRL_DIR exists: {OBJECTRL_DIR.exists()} -> {OBJECTRL_DIR}")
print(f"GYMNASIUM_DIR exists: {GYMNASIUM_DIR.exists()} -> {GYMNASIUM_DIR}")


## Runtime mode and experiment matrix

Use `DRY_RUN = True` until the ObjectRL command arguments and environment names have been verified. The full midway matrix is 3 algorithms x 3 environments x 5 seeds.


In [ ]:
ALGORITHMS = ["ppo", "sac", "td3"]
PROJECT_ENVIRONMENTS = [
    "car_racing_continuous",
    "cartpole_swingup",
    "acrobot_swingup",
]
SEEDS = [0, 1, 2, 3, 4]

RUN_CONFIG = {
    "debug": {
        "seeds": [0],
        "max_steps": 1_000,
        "eval_frequency": 250,
        "eval_episodes": 1,
        "save_frequency": 250,
        "device": "cpu",
    },
    "midway": {
        "seeds": SEEDS,
        "max_steps": 20_000,
        "eval_frequency": 2_000,
        "eval_episodes": 3,
        "save_frequency": 2_000,
        "device": "cpu",
    },
    "final": {
        "seeds": SEEDS,
        "max_steps": 500_000,
        "eval_frequency": 20_000,
        "eval_episodes": 5,
        "save_frequency": 20_000,
        "device": "cuda",
    },
}

RUN_MODE = "debug"  # debug | midway | final
DRY_RUN = True
STOP_ON_FAILURE = False

RUN_ORDER = [
    ("sac", "cartpole_swingup"),
    ("td3", "cartpole_swingup"),
    ("ppo", "cartpole_swingup"),
    ("sac", "acrobot_swingup"),
    ("td3", "acrobot_swingup"),
    ("ppo", "acrobot_swingup"),
    ("sac", "car_racing_continuous"),
    ("td3", "car_racing_continuous"),
    ("ppo", "car_racing_continuous"),
]

cfg = RUN_CONFIG[RUN_MODE]
print(f"RUN_MODE={RUN_MODE}, DRY_RUN={DRY_RUN}")
print(f"Planned jobs in this mode: {len(RUN_ORDER) * len(cfg['seeds'])}")


## ObjectRL config inspection

ObjectRL supports CLI arguments such as `--model.name`, `--env.name`, `--training.max-steps`, `--training.eval-episodes`, `--training.eval-frequency`, `--system.seed`, and `--logging.result-path`. The cells below inspect the local ObjectRL checkout rather than assuming config names blindly.


In [ ]:
def run_command(command: list[str], cwd: Path | None = None, timeout: int = 60) -> subprocess.CompletedProcess[str]:
    return subprocess.run(
        command,
        cwd=str(cwd) if cwd else None,
        capture_output=True,
        text=True,
        timeout=timeout,
    )


def grep_objectrl(pattern: str, max_lines: int = 80) -> str:
    if not OBJECTRL_DIR.exists():
        return f"ObjectRL directory not found: {OBJECTRL_DIR}"
    matches: list[str] = []
    regex = re.compile(pattern, flags=re.IGNORECASE)
    for path in sorted((OBJECTRL_DIR / "objectrl").rglob("*.py")):
        try:
            for line_no, line in enumerate(path.read_text(errors="ignore").splitlines(), start=1):
                if regex.search(line):
                    rel = path.relative_to(OBJECTRL_DIR)
                    matches.append(f"{rel}:{line_no}: {line}")
                    if len(matches) >= max_lines:
                        return "\n".join(matches)
        except OSError:
            continue
    return "\n".join(matches) if matches else f"No matches for pattern: {pattern}"


print("Model/env/seed/eval references:")
print(grep_objectrl(r"model\.name|env\.name|system\.seed|eval_frequency|eval_episodes|result_path", max_lines=120))


In [ ]:
# Make the local ObjectRL package importable for introspection in the notebook kernel.
if str(OBJECTRL_DIR) not in sys.path:
    sys.path.insert(0, str(OBJECTRL_DIR))


def safe_import_objectrl_configs() -> dict[str, Any]:
    try:
        from objectrl.config.config import EnvConfig, MainConfig, TrainingConfig
        from objectrl.config.model_configs import model_configs
        from objectrl.utils.make_env import dmc_mappings, env_mappings, gymnasium_mujoco_mappings

        return {
            "ok": True,
            "EnvConfig": EnvConfig,
            "MainConfig": MainConfig,
            "TrainingConfig": TrainingConfig,
            "model_configs": model_configs,
            "env_mappings": env_mappings,
            "dmc_mappings": dmc_mappings,
            "gymnasium_mujoco_mappings": gymnasium_mujoco_mappings,
        }
    except Exception as exc:  # Import can fail if ObjectRL dependencies are not installed.
        return {"ok": False, "error": repr(exc)}


objectrl_info = safe_import_objectrl_configs()
objectrl_info


In [ ]:
def discover_model_names() -> list[str]:
    if objectrl_info.get("ok"):
        return sorted(objectrl_info["model_configs"].keys())

    config_dir = OBJECTRL_DIR / "objectrl" / "config" / "model_configs"
    if not config_dir.exists():
        return []
    return sorted(path.stem for path in config_dir.glob("*.py") if path.stem != "__init__")


def discover_env_names() -> dict[str, str]:
    if objectrl_info.get("ok"):
        return dict(sorted(objectrl_info["env_mappings"].items()))

    make_env_path = OBJECTRL_DIR / "objectrl" / "utils" / "make_env.py"
    if not make_env_path.exists():
        return {}
    text = make_env_path.read_text(errors="ignore")
    discovered: dict[str, str] = {}
    for name in re.findall(r'"([A-Za-z0-9_\-]+)"\s*:', text):
        discovered[name] = "see objectrl/utils/make_env.py"
    return discovered


available_models = discover_model_names()
available_envs = discover_env_names()

print("Available ObjectRL model names:")
print(available_models)
print("\nProject baseline models present:")
print({algorithm: algorithm in available_models for algorithm in ALGORITHMS})
print("\nKnown ObjectRL environment aliases:")
for key, value in available_envs.items():
    print(f"  {key} -> {value}")


## Project environment mapping

The assignment requires continuous Car Racing, `cartpole-swingup-v0`, and `acrobot-swingup-v0`. ObjectRL's current `make_env.py` accepts Gymnasium MuJoCo aliases and a small DMC mapping such as `dmc-cheetah-run`, but it does not list these exact required environments in the inspected checkout.

The mapping below is therefore deliberately explicit and marked for verification. Keep `DRY_RUN = True` until these names are confirmed or wrappers are added outside `external/objectrl/`.


In [ ]:
PROJECT_ENV_METADATA = {
    "car_racing_continuous": {
        "display_name": "Car Racing continuous",
        "suite": "Gymnasium/Farama Box2D",
        "assignment_id": "CarRacing-v3 with continuous=True",
        "candidate_objectrl_env_name": "CarRacing-v3",
        "verification_status": "TODO_VERIFY_OBJECTRL_SUPPORT_OR_ADD_WRAPPER",
        "notes": "Gymnasium CarRacing uses continuous=True by default in recent versions; ObjectRL make_env currently does not map Box2D envs.",
    },
    "cartpole_swingup": {
        "display_name": "cartpole-swingup-v0",
        "suite": "DeepMind Control Suite",
        "assignment_id": "cartpole-swingup-v0",
        "candidate_objectrl_env_name": "dmc-cartpole-swingup",
        "verification_status": "TODO_VERIFY_DMC_MAPPING_OR_ADD_WRAPPER",
        "notes": "ObjectRL DMC naming convention is dmc-<domain>-<task>, but make_env currently lists only a subset of DMC tasks.",
    },
    "acrobot_swingup": {
        "display_name": "acrobot-swingup-v0",
        "suite": "DeepMind Control Suite",
        "assignment_id": "acrobot-swingup-v0",
        "candidate_objectrl_env_name": "dmc-acrobot-swingup",
        "verification_status": "TODO_VERIFY_DMC_MAPPING_OR_ADD_WRAPPER",
        "notes": "Verify whether dm_control exposes domain='acrobot', task='swingup' in the installed version.",
    },
}

OBJECTRL_ENV_NAMES = {
    key: value["candidate_objectrl_env_name"] for key, value in PROJECT_ENV_METADATA.items()
}

env_mapping_df = pd.DataFrame.from_dict(PROJECT_ENV_METADATA, orient="index")
env_mapping_df.index.name = "project_environment"
env_mapping_df


In [ ]:
def verify_project_env_mapping() -> pd.DataFrame:
    rows = []
    known_objectrl_values = set(available_envs.values()) | set(available_envs.keys())
    for project_env, meta in PROJECT_ENV_METADATA.items():
        candidate = meta["candidate_objectrl_env_name"]
        rows.append(
            {
                "project_environment": project_env,
                "assignment_id": meta["assignment_id"],
                "candidate_objectrl_env_name": candidate,
                "listed_by_objectrl": candidate in known_objectrl_values,
                "status": meta["verification_status"],
                "notes": meta["notes"],
            }
        )
    return pd.DataFrame(rows)


verification_df = verify_project_env_mapping()
verification_df


## Optional CLI help discovery

Run these cells if ObjectRL dependencies are installed. They print the non-model CLI and model-specific CLI help without launching training.


In [ ]:
def objectrl_main_path() -> Path:
    path = OBJECTRL_DIR / "objectrl" / "main.py"
    if not path.exists():
        raise FileNotFoundError(f"ObjectRL main.py not found at {path}")
    return path


def get_objectrl_help(model_name: str | None = None, max_chars: int = 6000) -> str:
    command = [sys.executable, str(objectrl_main_path()), "--help"]
    if model_name is not None:
        command = [sys.executable, str(objectrl_main_path()), "--help_model", model_name]
    result = run_command(command, cwd=OBJECTRL_DIR, timeout=60)
    text = result.stdout if result.stdout else result.stderr
    return text[:max_chars]


# Uncomment after installing ObjectRL dependencies in the active kernel environment.
# print(get_objectrl_help())
# for algorithm in ALGORITHMS:
#     print(f"\n--- {algorithm} model help ---")
#     print(get_objectrl_help(algorithm, max_chars=3000))


## Command generation

The generated command uses ObjectRL's documented CLI. The `--logging.result-path` points into this repository so ObjectRL's own `log.log`, `eval_results.npy`, and plots are kept under `results/raw/`.


In [ ]:
def shell_join(command: list[str]) -> str:
    return " ".join(shlex.quote(part) for part in command)


def run_name_for(algorithm: str, project_env_name: str, seed: int, mode: str) -> str:
    return f"{mode}_{algorithm}_{project_env_name}_seed{seed}"


def build_objectrl_command(
    algorithm: str,
    project_env_name: str,
    seed: int,
    max_steps: int,
    eval_frequency: int,
    eval_episodes: int,
    save_frequency: int,
    device: str,
    progress: bool = False,
    verbose: bool = True,
) -> tuple[list[str], str, Path]:
    if algorithm not in ALGORITHMS:
        raise ValueError(f"Unknown algorithm: {algorithm}")
    if project_env_name not in OBJECTRL_ENV_NAMES:
        raise ValueError(f"Unknown project environment: {project_env_name}")

    objectrl_env_name = OBJECTRL_ENV_NAMES[project_env_name]
    run_name = run_name_for(algorithm, project_env_name, seed, RUN_MODE)
    result_path = NOTEBOOK_RESULTS_DIR / run_name

    command = [
        sys.executable,
        str(objectrl_main_path()),
        "--model.name",
        algorithm,
        "--env.name",
        objectrl_env_name,
        "--training.max-steps",
        str(max_steps),
        "--training.eval-frequency",
        str(eval_frequency),
        "--training.eval-episodes",
        str(eval_episodes),
        "--training.no-parallelize-eval",
        "--logging.result-path",
        str(result_path),
        "--logging.save-frequency",
        str(save_frequency),
        "--system.seed",
        str(seed),
        "--system.device",
        device,
        "--system.storing-device",
        device,
    ]
    if progress:
        command.append("--progress")
    if verbose:
        command.append("--verbose")
    return command, run_name, result_path


example_command, example_run_name, example_result_path = build_objectrl_command(
    algorithm="sac",
    project_env_name="cartpole_swingup",
    seed=0,
    max_steps=cfg["max_steps"],
    eval_frequency=cfg["eval_frequency"],
    eval_episodes=cfg["eval_episodes"],
    save_frequency=cfg["save_frequency"],
    device=cfg["device"],
)
print(example_run_name)
print(shell_join(example_command))
print(example_result_path)


In [ ]:
def build_experiment_matrix(mode: str = RUN_MODE) -> pd.DataFrame:
    mode_cfg = RUN_CONFIG[mode]
    rows = []
    for algorithm, project_env_name in RUN_ORDER:
        for seed in mode_cfg["seeds"]:
            command, run_name, result_path = build_objectrl_command(
                algorithm=algorithm,
                project_env_name=project_env_name,
                seed=seed,
                max_steps=mode_cfg["max_steps"],
                eval_frequency=mode_cfg["eval_frequency"],
                eval_episodes=mode_cfg["eval_episodes"],
                save_frequency=mode_cfg["save_frequency"],
                device=mode_cfg["device"],
            )
            rows.append(
                {
                    "run_mode": mode,
                    "algorithm": algorithm,
                    "environment": project_env_name,
                    "display_environment": PROJECT_ENV_METADATA[project_env_name]["display_name"],
                    "objectrl_env_name": OBJECTRL_ENV_NAMES[project_env_name],
                    "seed": seed,
                    "max_steps": mode_cfg["max_steps"],
                    "eval_frequency": mode_cfg["eval_frequency"],
                    "eval_episodes": mode_cfg["eval_episodes"],
                    "run_name": run_name,
                    "result_path": str(result_path),
                    "command": shell_join(command),
                }
            )
    return pd.DataFrame(rows)


matrix_df = build_experiment_matrix(RUN_MODE)
matrix_csv = PROCESSED_RESULTS_DIR / f"midway_poc_copilotcli_matrix_{RUN_MODE}.csv"
matrix_df.to_csv(matrix_csv, index=False)
print(f"Wrote matrix: {matrix_csv}")
matrix_df


## Dry-run and subprocess execution

The runner always writes a per-run status JSON and captures subprocess stdout/stderr under `results/logs/`. Keep `DRY_RUN = True` while environment mappings are unresolved.


In [ ]:
def run_experiment(row: pd.Series, dry_run: bool = DRY_RUN) -> dict[str, Any]:
    command, run_name, result_path = build_objectrl_command(
        algorithm=row["algorithm"],
        project_env_name=row["environment"],
        seed=int(row["seed"]),
        max_steps=int(row["max_steps"]),
        eval_frequency=int(row["eval_frequency"]),
        eval_episodes=int(row["eval_episodes"]),
        save_frequency=RUN_CONFIG[row["run_mode"]]["save_frequency"],
        device=RUN_CONFIG[row["run_mode"]]["device"],
    )
    result_path.mkdir(parents=True, exist_ok=True)

    status_file = result_path / "status.json"
    console_log_file = LOGS_DIR / f"{run_name}.console.log"

    status: dict[str, Any] = {
        "run_mode": row["run_mode"],
        "algorithm": row["algorithm"],
        "environment": row["environment"],
        "display_environment": row["display_environment"],
        "objectrl_env_name": row["objectrl_env_name"],
        "seed": int(row["seed"]),
        "max_steps": int(row["max_steps"]),
        "eval_frequency": int(row["eval_frequency"]),
        "eval_episodes": int(row["eval_episodes"]),
        "run_name": run_name,
        "result_path": str(result_path),
        "console_log_file": str(console_log_file),
        "command": shell_join(command),
        "status": "not_started",
        "started_at_unix": None,
        "finished_at_unix": None,
        "duration_seconds": None,
        "return_code": None,
    }

    if dry_run:
        status["status"] = "dry_run"
        status_file.write_text(json.dumps(status, indent=2))
        return status

    start = time.time()
    status["started_at_unix"] = start
    with console_log_file.open("w") as log_handle:
        process = subprocess.run(
            command,
            cwd=str(OBJECTRL_DIR),
            stdout=log_handle,
            stderr=subprocess.STDOUT,
            text=True,
        )
    end = time.time()

    status["finished_at_unix"] = end
    status["duration_seconds"] = end - start
    status["return_code"] = process.returncode
    status["status"] = "completed" if process.returncode == 0 else "failed"
    status_file.write_text(json.dumps(status, indent=2))
    return status


def run_matrix(matrix: pd.DataFrame, dry_run: bool = DRY_RUN) -> pd.DataFrame:
    statuses = []
    for _, row in matrix.iterrows():
        status = run_experiment(row, dry_run=dry_run)
        statuses.append(status)
        print(f"{status['status']:>10} | {status['run_name']}")
        if status["status"] == "failed" and STOP_ON_FAILURE:
            break
    status_df = pd.DataFrame(statuses)
    status_csv = PROCESSED_RESULTS_DIR / f"midway_poc_copilotcli_status_{RUN_MODE}.csv"
    status_json = PROCESSED_RESULTS_DIR / f"midway_poc_copilotcli_status_{RUN_MODE}.json"
    status_df.to_csv(status_csv, index=False)
    status_json.write_text(json.dumps(statuses, indent=2))
    print(f"Wrote status CSV: {status_csv}")
    print(f"Wrote status JSON: {status_json}")
    return status_df


# Execute the configured matrix. With DRY_RUN=True this writes commands/status only.
status_df = run_matrix(matrix_df, dry_run=DRY_RUN)
status_df


## Evaluation-return parsing

ObjectRL writes evaluation results to timestamped run directories under `logging.result_path / env / model / seed_xx / timestamp/`. This notebook first tries to read `eval_results.npy`, then falls back to `log.log` lines such as:

```text
EVALUATION    N-steps:       0    Mean_Reward:      1.234    IQM_Reward:      1.234
```

The parsed tidy format is `algorithm, environment, seed, step, evaluation_return`.


In [ ]:
def find_objectrl_run_dirs(result_path: Path) -> list[Path]:
    if not result_path.exists():
        return []
    candidates = [p.parent for p in result_path.rglob("log.log")]
    candidates.extend(p.parent for p in result_path.rglob("eval_results.npy"))
    return sorted(set(candidates))


def parse_eval_results_npy(eval_path: Path) -> list[dict[str, float | int]]:
    rows: list[dict[str, float | int]] = []
    try:
        data = np.load(eval_path, allow_pickle=True).item()
    except Exception:
        return rows
    if not isinstance(data, dict):
        return rows
    for step, rewards in sorted(data.items(), key=lambda item: int(item[0])):
        rewards_array = np.asarray(rewards, dtype=float)
        rows.append(
            {
                "step": int(step),
                "evaluation_return": float(np.mean(rewards_array)),
                "evaluation_return_std_within_eval": float(np.std(rewards_array)),
                "evaluation_episodes": int(rewards_array.size),
                "parser_source": str(eval_path),
            }
        )
    return rows


def parse_objectrl_log(log_path: Path) -> list[dict[str, float | int | str]]:
    patterns = [
        re.compile(
            r"EVALUATION\s+N-steps:\s*(?P<step>\d+)\s+Mean_Reward:\s*(?P<ret>-?\d+(?:\.\d+)?)",
            flags=re.IGNORECASE,
        ),
        re.compile(
            r"step[:=]\s*(?P<step>\d+).*eval.*(?:return|reward)[:=]\s*(?P<ret>-?\d+(?:\.\d+)?)",
            flags=re.IGNORECASE,
        ),
    ]
    rows: list[dict[str, float | int | str]] = []
    try:
        lines = log_path.read_text(errors="ignore").splitlines()
    except OSError:
        return rows
    for line in lines:
        for pattern in patterns:
            match = pattern.search(line)
            if match:
                rows.append(
                    {
                        "step": int(match.group("step")),
                        "evaluation_return": float(match.group("ret")),
                        "evaluation_return_std_within_eval": np.nan,
                        "evaluation_episodes": np.nan,
                        "parser_source": str(log_path),
                    }
                )
                break
    return rows


def parse_run_evaluations(status: dict[str, Any]) -> list[dict[str, Any]]:
    result_path = Path(status["result_path"])
    parsed_rows: list[dict[str, Any]] = []
    for run_dir in find_objectrl_run_dirs(result_path):
        rows = parse_eval_results_npy(run_dir / "eval_results.npy")
        if not rows:
            rows = parse_objectrl_log(run_dir / "log.log")
        for row in rows:
            parsed_rows.append(
                {
                    "run_mode": status["run_mode"],
                    "algorithm": status["algorithm"],
                    "environment": status["environment"],
                    "display_environment": status["display_environment"],
                    "objectrl_env_name": status["objectrl_env_name"],
                    "seed": status["seed"],
                    "run_name": status["run_name"],
                    **row,
                }
            )
    return parsed_rows


def collect_evaluation_returns(status_df: pd.DataFrame) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for status in status_df.to_dict(orient="records"):
        rows.extend(parse_run_evaluations(status))
    columns = [
        "run_mode",
        "algorithm",
        "environment",
        "display_environment",
        "objectrl_env_name",
        "seed",
        "run_name",
        "step",
        "evaluation_return",
        "evaluation_return_std_within_eval",
        "evaluation_episodes",
        "parser_source",
    ]
    return pd.DataFrame(rows, columns=columns)


evaluation_df = collect_evaluation_returns(status_df)
evaluation_csv = PROCESSED_RESULTS_DIR / f"midway_poc_copilotcli_evaluation_returns_{RUN_MODE}.csv"
evaluation_df.to_csv(evaluation_csv, index=False)
print(f"Parsed {len(evaluation_df)} evaluation rows")
print(f"Wrote evaluation CSV: {evaluation_csv}")
evaluation_df.head()


## Aggregation and learning-curve export

The midway report needs one learning-curve figure with environments as subplots and algorithms as curves. If no completed evaluation rows exist yet, the plotting function writes a placeholder figure that clearly states that runs are pending instead of fabricating data.


In [ ]:
def summarize_evaluations(evaluation_df: pd.DataFrame) -> pd.DataFrame:
    if evaluation_df.empty:
        return pd.DataFrame(
            columns=[
                "run_mode",
                "algorithm",
                "environment",
                "display_environment",
                "step",
                "mean_return",
                "std_return",
                "n_seeds",
            ]
        )
    return (
        evaluation_df.groupby(
            ["run_mode", "algorithm", "environment", "display_environment", "step"],
            as_index=False,
        )
        .agg(
            mean_return=("evaluation_return", "mean"),
            std_return=("evaluation_return", "std"),
            n_seeds=("seed", "nunique"),
        )
        .sort_values(["environment", "algorithm", "step"])
    )


summary_df = summarize_evaluations(evaluation_df)
summary_csv = PROCESSED_RESULTS_DIR / f"midway_poc_copilotcli_evaluation_summary_{RUN_MODE}.csv"
summary_df.to_csv(summary_csv, index=False)
print(f"Wrote summary CSV: {summary_csv}")
summary_df.head()


In [ ]:
def plot_learning_curves(summary_df: pd.DataFrame, output_base: Path) -> plt.Figure:
    environments = PROJECT_ENVIRONMENTS
    fig, axes = plt.subplots(1, len(environments), figsize=(5.5 * len(environments), 4.2), sharey=False)
    if len(environments) == 1:
        axes = [axes]

    if summary_df.empty:
        for ax, env_name in zip(axes, environments):
            ax.set_title(PROJECT_ENV_METADATA[env_name]["display_name"])
            ax.set_xlabel("Training steps before evaluation")
            ax.set_ylabel("Undiscounted evaluation episode return")
            ax.text(
                0.5,
                0.5,
                "No parsed evaluation results yet\n(run with DRY_RUN=False after verifying env IDs)",
                ha="center",
                va="center",
                transform=ax.transAxes,
            )
            ax.grid(True, alpha=0.3)
    else:
        for ax, env_name in zip(axes, environments):
            env_df = summary_df[summary_df["environment"] == env_name]
            ax.set_title(PROJECT_ENV_METADATA[env_name]["display_name"])
            ax.set_xlabel("Training steps before evaluation")
            ax.set_ylabel("Undiscounted evaluation episode return")
            ax.grid(True, alpha=0.3)

            for algorithm in ALGORITHMS:
                alg_df = env_df[env_df["algorithm"] == algorithm].sort_values("step")
                if alg_df.empty:
                    continue
                x = alg_df["step"].to_numpy(dtype=float)
                y = alg_df["mean_return"].to_numpy(dtype=float)
                ax.plot(x, y, marker="o", label=algorithm.upper())
                if alg_df["std_return"].notna().any():
                    std = alg_df["std_return"].fillna(0).to_numpy(dtype=float)
                    ax.fill_between(x, y - std, y + std, alpha=0.2)
            ax.legend()

    fig.suptitle("DM887 midway baseline learning curves: PPO, SAC, TD3", y=1.02)
    fig.tight_layout()
    output_base.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_base.with_suffix(".pdf"), bbox_inches="tight")
    fig.savefig(output_base.with_suffix(".png"), dpi=200, bbox_inches="tight")
    return fig


figure_base = FIGURES_DIR / f"midway_poc_copilotcli_baselines_{RUN_MODE}"
fig = plot_learning_curves(summary_df, figure_base)
print(f"Wrote {figure_base.with_suffix('.pdf')}")
print(f"Wrote {figure_base.with_suffix('.png')}")
plt.show()


## Report-facing notes

Use this cell to generate concise status text for the midway report. Keep failed or missing runs visible; do not present dry-run output as experimental evidence.


In [ ]:
def report_status_notes(status_df: pd.DataFrame, evaluation_df: pd.DataFrame) -> str:
    counts = status_df.groupby(["algorithm", "environment", "status"]).size().reset_index(name="count")
    completed = status_df[status_df["status"] == "completed"]
    failed = status_df[status_df["status"] == "failed"]
    dry = status_df[status_df["status"] == "dry_run"]

    lines = [
        "### Midway baseline pipeline status",
        "",
        f"Run mode: `{RUN_MODE}`.",
        f"Dry-run mode: `{DRY_RUN}`.",
        f"Planned runs in this mode: {len(status_df)}.",
        f"Completed subprocess runs: {len(completed)}.",
        f"Failed subprocess runs: {len(failed)}.",
        f"Dry-run command records: {len(dry)}.",
        f"Parsed evaluation rows: {len(evaluation_df)}.",
        "",
        "The pipeline defines PPO, SAC, and TD3 ObjectRL baselines for continuous Car Racing, cartpole-swingup-v0, and acrobot-swingup-v0 with seeds 0--4. It saves per-run logs/status files, parses ObjectRL evaluation returns when available, aggregates by algorithm/environment/seed/step, and exports matplotlib learning curves under `figures/midway/`.",
        "",
        "TODO before full experiments: verify or implement repository-local wrappers for the exact assignment environments, because the inspected ObjectRL checkout does not list continuous Car Racing, dmc-cartpole-swingup, or dmc-acrobot-swingup in its built-in environment mapping.",
        "",
        "Status counts:",
        counts.to_string(index=False) if not counts.empty else "No status rows available.",
    ]
    if not failed.empty:
        lines.extend([
            "",
            "Failed runs and console logs:",
            failed[["algorithm", "environment", "seed", "console_log_file"]].to_string(index=False),
        ])
    return "\n".join(lines)


notes = report_status_notes(status_df, evaluation_df)
print(notes)
notes_path = PROCESSED_RESULTS_DIR / f"midway_poc_copilotcli_report_notes_{RUN_MODE}.md"
notes_path.write_text(notes)
print(f"\nWrote report notes: {notes_path}")


## Manual verification checklist

Before switching `DRY_RUN` to `False` for the midway/final matrix:

- Confirm ObjectRL imports in the active kernel environment.
- Confirm `ppo`, `sac`, and `td3` appear in discovered ObjectRL model names.
- Confirm or implement support for the exact required environments without modifying `external/objectrl/`:
  - continuous `CarRacing-v*` from Gymnasium/Farama Box2D,
  - DMC `cartpole`/`swingup`,
  - DMC `acrobot`/`swingup`.
- Update `OBJECTRL_ENV_NAMES` if actual IDs differ.
- Run a single debug command first and inspect `results/logs/*.console.log` plus ObjectRL's `log.log`.
- Verify that parsed `evaluation_return` is the undiscounted evaluation episode return.
- Only then run the five-seed midway or final matrix.
